# Offset-Free MPC Steady-State Debug

This notebook keeps the normal disturbance offset-free MPC run intact and computes the frozen-`dhat` steady-state targets as an analysis-only sidecar.

In [ ]:
from IPython.display import display

import numpy as np
import os

from Simulation.system_functions import PolymerCSTR
from Simulation.mpc import MpcSolver, compute_observer_gain
from Simulation.mpc_run import run_mpc
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from analysis.steady_state_debug_analysis import (
    analyze_offsetfree_rollout,
    run_synthetic_smoke_checks,
    save_offsetfree_ss_debug_artifacts,
)
from utils.scaling_helpers import apply_min_max
from utils.td3_helpers import load_and_prepare_system_data

In [ ]:
# System and steady-state setup
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr.ss_inputs, "y_ss": cstr.y_ss}
dir_path = os.path.join(os.getcwd(), "Data")

In [ ]:
# Identified model, scaling, observer, and MPC configuration
setpoint_y = np.array([[2.8, 320.0], [5.0, 326.0]])
u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324.0], [3.4, 321.0]])
y_sp_scenario = (
    apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
    - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
)

poles = np.array([
    0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
    0.42900673, 0.4228262, 0.96916776, 0.91230187,
])
L = compute_observer_gain(A_aug, C_aug, poles)

n_inputs = 2
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78.0]), data_min[:n_inputs], data_max[:n_inputs])
b_max = apply_min_max(np.array([870.0, 670.0]), data_min[:n_inputs], data_max[:n_inputs])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
cons = []
IC_opt = np.zeros(n_inputs * cont_h)

MPC_obj = MpcSolver(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([5.0, 1.0]),
    R_in=np.array([1.0, 1.0]),
    NP=predict_h,
    NC=cont_h,
)

_, reward_fn_qp = make_reward_fn_mpc_quadratic(
    Q_diag=np.array([5.0, 1.0]),
    R_diag=np.array([1.0, 1.0]),
)

n_tests = 2
set_points_len = 400
TEST_CYCLE = [False, False]
warm_start = 0
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

In [ ]:
# Disturbance run only; controller behavior is unchanged
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
(y_mpc, u_mpc, avg_rewards, rewards, xhatdhat, nFE,
 time_in_sub_episodes, y_sp, yhat, delta_y_storage, delta_u_storage) = run_mpc(
    cstr,
    MPC_obj,
    y_sp_scenario,
    n_tests,
    set_points_len,
    steady_states,
    IC_opt,
    bnds,
    cons,
    warm_start,
    L,
    data_min,
    data_max,
    TEST_CYCLE,
    reward_fn_qp,
    nominal_qi,
    nominal_qs,
    nominal_hA,
    qi_change,
    qs_change,
    ha_change,
    mode="disturb",
)

In [ ]:
# Sidecar steady-state analysis and artifact export
rollout = {
    "y_mpc": y_mpc,
    "u_mpc": u_mpc,
    "xhatdhat": xhatdhat,
    "y_sp": y_sp,
    "nFE": nFE,
    "delta_t": delta_t,
}
analysis_config = {
    "enabled": True,
    "solver_mode": "auto",
    "cond_warn_threshold": 1.0e8,
    "residual_warn_threshold": 1.0e-8,
    "enable_box_analysis": True,
    "box_bound_tol": 1.0e-9,
    "box_use_reduced_first": True,
    "box_event_window_radius": 5,
    "box_dhat_event_threshold": 5.0e-2,
    "box_max_event_plots": 6,
    "save_csv": True,
    "save_plots": True,
    "sample_table_stride": 20,
    "case_name": "disturbance",
}

bundle = analyze_offsetfree_rollout(
    rollout=rollout,
    system_data=system_data,
    steady_states=steady_states,
    analysis_config=analysis_config,
)
out_dir = save_offsetfree_ss_debug_artifacts(bundle, directory=dir_path)
print("Saved debug artifacts to:", out_dir)

smoke_checks = run_synthetic_smoke_checks()
print("Synthetic smoke checks:", smoke_checks)

In [ ]:
# Compact in-notebook summary
try:
    import pandas as pd

    display(pd.DataFrame([bundle["model_structure"]]))
    display(pd.DataFrame([bundle["linear_algebra_summary"]]))
    display(pd.DataFrame(bundle["sampled_step_rows"]))
    if bundle.get("box_analysis_enabled"):
        display(pd.DataFrame([bundle["box_analysis"]["overall_summary"]]))
        display(pd.DataFrame(bundle["box_analysis"]["per_input_rows"]))
        display(pd.DataFrame(bundle["box_analysis"]["event_rows"]).head(20))
except Exception:
    print(bundle["model_structure"])
    print(bundle["linear_algebra_summary"])
    print(bundle["sampled_step_rows"][:5])
    if bundle.get("box_analysis_enabled"):
        print(bundle["box_analysis"]["overall_summary"])
        print(bundle["box_analysis"]["per_input_rows"][:2])
        print(bundle["box_analysis"]["event_rows"][:5])